In [1]:

import sys

from pathlib import Path

print(sys.executable)

print(Path.cwd())

c:\Users\hiju2\AppData\Local\Programs\Python\Python314\python.exe
c:\dev\ai-data-analysis\chapter09


### STEP 1. 예측 문제와 예측 시점을 먼저 정의합니다

무엇을 예측하는가?

예측 단위는 무엇인가?

언제 예측하는가?

그 시점에 실제로 알 수 있는 정보는 무엇인가?

STEP 2는 AI가 답을 미리 보고 문제를 푸는 것을 막는 단계입니다.
예를 들어 시험을 보는데 문제 옆에 정답이 적혀 있다면 누구나 높은 점수를 받을 수 있습니다. AI도 마찬가지입니다. 예측하려는 매출과 직접 관련된 order_total, quantity 같은 정보를 미리 주면 정확도가 비정상적으로 높아집니다. 그래서 예측하기 전에 알 수 있는 정보만 골라서 사용하는 과정입니다.

In [2]:
import pandas as pd
from pathlib import Path

# 1. processed 폴더의 실제 CSV 파일 불러오기
processed_path = Path("../data/processed")

csv_files = list(processed_path.glob("*.csv"))

dataframes = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

# 2. 실제 데이터에 존재하는 모든 컬럼 수집
all_features = sorted(
    set(
        col
        for df in dataframes.values()
        for col in df.columns
    )
)

# 3. Feature Leakage 판단 기준
allowed_features = {
    "order_month",
    "order_dayofweek",
    "age",
    "payment_method",
    "gender",
    "city",
}

target_materials = {
    "order_total",
    "line_total",
    "quantity",
    "unit_price",
    "item_count",
    "total_quantity",
    "avg_unit_price",
}

post_or_id = {
    "order_status",
    "order_id",
    "customer_id",
    "product_id",
}

# 4. 실제 존재하는 컬럼만 평가
rows = []

for feature in all_features:

    if feature in allowed_features:
        available = "O"
        target = "X"
        post_id = "X"
        final = "O"
        reason = "예측 시점에 알 수 있는 정보"

    elif feature in target_materials:
        available = "X"
        target = "O"
        post_id = "X"
        final = "X"
        reason = "Target 계산에 사용되거나 직접 관련된 정보"

    elif feature in post_or_id:
        available = "X"
        target = "X"
        post_id = "O"
        final = "X"
        reason = "사후 정보이거나 식별용 ID"

    else:
        available = "확인 필요"
        target = "확인 필요"
        post_id = "확인 필요"
        final = "확인 필요"
        reason = "현재 기준에 없으므로 직접 확인 필요"

    rows.append([
        feature,
        available,
        target,
        post_id,
        final,
        reason
    ])

# 5. 표 생성
leakage_audit = pd.DataFrame(
    rows,
    columns=[
        "feature",
        "예측 시점 사용 가능",
        "Target 계산 재료",
        "사후 정보/ID",
        "최종 사용",
        "판단 이유",
    ]
)

print("불러온 파일:", [f.name for f in csv_files])
display(leakage_audit)

불러온 파일: ['customers_clean.csv', 'orders_clean.csv', 'order_items_clean.csv', 'products_clean.csv', 'visualization.csv']


,feature,예측 시점 사용 가능,Target 계산 재료,사후 정보/ID,최종 사용,판단 이유
0,age,O,X,X,O,예측 시점에 알 수 있는 정보
1,category,확인 필요,확인 필요,확인 필요,확인 필요,현재 기준에 없으므로 직접 확인 필요
2,city,O,X,X,O,예측 시점에 알 수 있는 정보
3,customer_id,X,X,O,X,사후 정보이거나 식별용 ID
4,customer_match,확인 필요,확인 필요,확인 필요,확인 필요,현재 기준에 없으므로 직접 확인 필요
5,gender,O,X,X,O,예측 시점에 알 수 있는 정보
6,line_total,X,O,X,X,Target 계산에 사용되거나 직접 관련된 정보
7,merge_valid,확인 필요,확인 필요,확인 필요,확인 필요,현재 기준에 없으므로 직접 확인 필요
8,name,확인 필요,확인 필요,확인 필요,확인 필요,현재 기준에 없으므로 직접 확인 필요
9,order_date,확인 필요,확인 필요,확인 필요,확인 필요,현재 기준에 없으므로 직접 확인 필요


### Feature Leakage Audit 결과 요약

총 24개의 feature를 확인했습니다.

- **최종 사용 가능:** `age`, `city`, `gender`, `order_dayofweek`, `order_month`, `payment_method`
  - 예측 시점에 알 수 있고 Target 계산에 직접 사용되지 않는 정보이므로 사용합니다.

- **Target 계산과 관련되어 제외:** `line_total`, `quantity`, `unit_price`
  - 예측하려는 값과 직접적으로 관련된 정보이므로 데이터 누수(Leakage)를 막기 위해 제외합니다.

- **사후 정보 또는 ID라서 제외:** `customer_id`, `order_id`, `order_status`, `product_id`
  - 단순 식별값이거나 주문 이후 결정되는 정보이므로 제외합니다.

- **추가 확인 필요:** `category`, `customer_match`, `merge_valid`, `name`, `order_date`, `order_item_id`, `order_match`, `price`, `product_match`, `product_name`, `signup_date`
  - 현재 판단 기준에 포함되지 않아 데이터의 의미와 생성 시점을 추가로 확인해야 합니다.

**결론:** 현재 기준으로 머신러닝 학습에 사용할 feature는 총 6개이며, 나머지 변수는 데이터 누수 가능성 또는 변수의 의미를 확인한 뒤 사용 여부를 결정해야 합니다.

In [3]:
import pandas as pd
from pathlib import Path

# =========================================================
# STEP 2. Feature Leakage Audit
# Target: order_total
# 예측 단위: 주문 1건
#
# order_total = 같은 order_id의 line_total 합계
# line_total = quantity × unit_price
# =========================================================

# 1. processed 폴더의 실제 CSV 파일 읽기
processed_path = Path("../data/processed")

csv_files = list(processed_path.glob("*.csv"))

dataframes = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

# 실제 존재하는 모든 컬럼
all_features = sorted({
    col
    for df in dataframes.values()
    for col in df.columns
})

# 2. Feature별 판단 기준
audit_rules = {
    # 최종 사용
    "age": (
        "O", "X", "X", "O",
        "주문 전에 알 수 있는 고객의 비식별 특성"
    ),
    "gender": (
        "O", "X", "X", "O",
        "주문 전에 알 수 있는 고객의 비식별 특성"
    ),
    "city": (
        "O", "X", "X", "O",
        "주문 전에 알 수 있는 고객의 지역 정보"
    ),
    "payment_method": (
        "O", "X", "X", "O",
        "주문 메타데이터로 예측 시점에 알 수 있음"
    ),
    "order_dayofweek": (
        "O", "X", "X", "O",
        "주문 시점의 요일 정보로 Target 계산과 무관"
    ),
    "order_month": (
        "O", "X", "X", "O",
        "사용 가능하나 YYYY-MM 대신 월(1~12)로 변환 권장"
    ),

    # Target 또는 Target 계산 재료 → 누수
    "order_total": (
        "X", "O", "X", "X",
        "예측하려는 Target 자체"
    ),
    "line_total": (
        "X", "O", "X", "X",
        "order_total을 계산하는 직접 재료"
    ),
    "quantity": (
        "X", "O", "X", "X",
        "line_total = quantity × unit_price이므로 Target 계산 재료"
    ),
    "unit_price": (
        "X", "O", "X", "X",
        "line_total = quantity × unit_price이므로 Target 계산 재료"
    ),
    "price": (
        "X", "O", "X", "X",
        "unit_price와 거의 동일하여 Target 정보가 누수될 위험이 큼"
    ),
    "item_count": (
        "X", "O", "X", "X",
        "주문 상세 결과에서 계산되는 정보"
    ),
    "total_quantity": (
        "X", "O", "X", "X",
        "주문 상세 수량을 집계한 정보"
    ),
    "avg_unit_price": (
        "X", "O", "X", "X",
        "주문 상세 단가에서 계산되는 정보"
    ),

    # ID / 개인정보 / 사후 정보
    "order_id": (
        "O", "X", "O", "X",
        "주문 식별용 ID이므로 모델 입력에서 제외"
    ),
    "customer_id": (
        "O", "X", "O", "X",
        "고객 식별용 ID이므로 모델 입력에서 제외"
    ),
    "product_id": (
        "X", "X", "O", "X",
        "상품 식별용 ID이며 주문 상세 정보"
    ),
    "order_item_id": (
        "X", "X", "O", "X",
        "주문 상세 행을 구분하는 ID"
    ),
    "name": (
        "O", "X", "O", "X",
        "고객 개인 식별 정보이므로 제외"
    ),
    "product_name": (
        "X", "X", "O", "X",
        "상품을 사실상 식별하는 정보이므로 제외"
    ),
    "order_status": (
        "X", "X", "O", "X",
        "주문 이후 결정될 수 있는 사후 정보"
    ),

    # 병합 검증용 컬럼
    "order_match": (
        "X", "X", "O", "X",
        "데이터 병합 검증용 컬럼이므로 학습에 사용하지 않음"
    ),
    "product_match": (
        "X", "X", "O", "X",
        "데이터 병합 검증용 컬럼이므로 학습에 사용하지 않음"
    ),
    "customer_match": (
        "X", "X", "O", "X",
        "데이터 병합 검증용 컬럼이므로 학습에 사용하지 않음"
    ),
    "merge_valid": (
        "X", "X", "O", "X",
        "병합 오류 행을 제거하는 데만 사용하고 학습에서는 제외"
    ),

    # 원본 날짜
    "order_date": (
        "O", "X", "X", "X",
        "원본 날짜 대신 월·요일 등 파생변수를 사용"
    ),

    # 변환 후 사용
    "signup_date": (
        "O", "X", "X", "변환 후 O",
        "원본 대신 주문일-가입일로 계산한 '가입 후 경과일' 사용"
    ),

    # 조건부
    "category": (
        "조건부 O", "X", "X", "조건부 O",
        "예측 시점에 구매 상품 카테고리를 알고 있을 때만 사용"
    ),
}

# 3. 실제 데이터에 존재하는 컬럼만 표에 추가
rows = []

for feature in all_features:
    if feature in audit_rules:
        available, target_material, post_id, final_use, reason = audit_rules[feature]
    else:
        available = "확인 필요"
        target_material = "확인 필요"
        post_id = "확인 필요"
        final_use = "확인 필요"
        reason = "현재 기준에 없는 컬럼으로 의미 확인 필요"

    rows.append([
        feature,
        available,
        target_material,
        post_id,
        final_use,
        reason
    ])

# 4. DataFrame 생성
leakage_audit = pd.DataFrame(
    rows,
    columns=[
        "feature",
        "예측 시점 사용 가능",
        "Target 계산 재료",
        "사후 정보/ID",
        "최종 사용",
        "판단 이유"
    ]
)

print("Target: order_total")
print("예측 단위: 주문 1건")
print("불러온 CSV:", [file.name for file in csv_files])
print("실제 컬럼 수:", len(all_features))

display(leakage_audit)

Target: order_total
예측 단위: 주문 1건
불러온 CSV: ['customers_clean.csv', 'orders_clean.csv', 'order_items_clean.csv', 'products_clean.csv', 'visualization.csv']
실제 컬럼 수: 24


,feature,예측 시점 사용 가능,Target 계산 재료,사후 정보/ID,최종 사용,판단 이유
0,age,O,X,X,O,주문 전에 알 수 있는 고객의 비식별 특성
1,category,조건부 O,X,X,조건부 O,예측 시점에 구매 상품 카테고리를 알고 있을 때만 사용
2,city,O,X,X,O,주문 전에 알 수 있는 고객의 지역 정보
3,customer_id,O,X,O,X,고객 식별용 ID이므로 모델 입력에서 제외
4,customer_match,X,X,O,X,데이터 병합 검증용 컬럼이므로 학습에 사용하지 않음
5,gender,O,X,X,O,주문 전에 알 수 있는 고객의 비식별 특성
6,line_total,X,O,X,X,order_total을 계산하는 직접 재료
7,merge_valid,X,X,O,X,병합 오류 행을 제거하는 데만 사용하고 학습에서는 제외
8,name,O,X,O,X,고객 개인 식별 정보이므로 제외
9,order_date,O,X,X,X,원본 날짜 대신 월·요일 등 파생변수를 사용


### Feature Leakage Audit 결과 요약

이번 회귀 분석의 **Target은 `order_total`**이며, **주문 1건의 총금액을 예측**하는 것이 목표입니다.

- `order_total`은 같은 `order_id`의 `line_total`을 합한 값입니다.
- `line_total = quantity × unit_price`이므로 `line_total`, `quantity`, `unit_price`는 Target을 계산하는 정보라서 제외합니다.
- `price`도 `unit_price`와 거의 같은 정보이므로 데이터 누수를 막기 위해 제외합니다.
- `order_id`, `customer_id`, `product_id`, `order_item_id` 등 ID 컬럼은 모델 학습에 사용하지 않습니다.
- `order_status`는 주문 이후 알게 되는 사후 정보이므로 제외합니다.
- `order_match`, `product_match`, `customer_match`, `merge_valid`는 데이터 검증용 컬럼이므로 학습에서 제외합니다.
- `order_date`는 그대로 사용하지 않고 `order_month`, `order_dayofweek`와 같은 파생변수로 사용합니다.
- `signup_date`는 `가입 후 경과일`로 변환한 뒤 사용할 수 있습니다.
- `category`는 예측 시점에 상품 카테고리를 알고 있다는 조건에서만 사용할 수 있습니다.

**최종적으로 사용할 기본 Feature 후보는 `age`, `gender`, `city`, `payment_method`, `order_dayofweek`, `order_month`이며, 변환 후 `가입 후 경과일`, 조건에 따라 `category`를 추가할 수 있습니다.**

> 즉, 정답인 주문 총금액을 미리 알 수 있게 해주는 정보는 제외하고, 실제 예측 시점에 알 수 있는 정보만 모델에 사용합니다.

### STEP 3. Target 생성과 관계 검증을 확인합니다

Target은 주문 상세 데이터에서 만듭니다.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# STEP 3. Target 생성과 관계 검증
# Target: order_total
# 예측 단위: 주문 1건
# =========================================================

# 1. 데이터 불러오기
processed_path = Path("../data/processed")

orders = pd.read_csv(processed_path / "orders_clean.csv")
order_items = pd.read_csv(processed_path / "order_items_clean.csv")
customers = pd.read_csv(processed_path / "customers_clean.csv")


# ---------------------------------------------------------
# 2. line_total 재계산 및 기존 값과 비교
# line_total = quantity × unit_price
# ---------------------------------------------------------

order_items["line_total_calc"] = (
    order_items["quantity"] * order_items["unit_price"]
)

if "line_total" in order_items.columns:
    line_total_mismatch = ~np.isclose(
        order_items["line_total"],
        order_items["line_total_calc"],
        equal_nan=True
    )

    line_total_mismatch_count = line_total_mismatch.sum()
else:
    line_total_mismatch_count = 0
    print("※ 기존 line_total 컬럼이 없어 계산값만 사용합니다.")


# ---------------------------------------------------------
# 3. Target 생성
# order_total = order_id별 line_total 합계
# 반드시 직접 계산한 line_total_calc 사용
# ---------------------------------------------------------

order_total = (
    order_items
    .groupby("order_id", as_index=False)["line_total_calc"]
    .sum()
    .rename(columns={"line_total_calc": "order_total"})
)


# ---------------------------------------------------------
# 4. orders ↔ target 관계 검증
# 기대 관계: one_to_one
# ---------------------------------------------------------

duplicate_orders = orders["order_id"].duplicated().sum()
duplicate_targets = order_total["order_id"].duplicated().sum()

orders_without_target = (
    ~orders["order_id"].isin(order_total["order_id"])
).sum()

targets_without_order = (
    ~order_total["order_id"].isin(orders["order_id"])
).sum()

orders_target_relation_ok = (
    duplicate_orders == 0
    and duplicate_targets == 0
    and orders_without_target == 0
    and targets_without_order == 0
)

if orders_target_relation_ok:
    orders_target_result = "PASS (one_to_one)"
else:
    orders_target_result = "FAIL"


# ---------------------------------------------------------
# 5. orders → customers 관계 검증
# 기대 관계: many_to_one
# ---------------------------------------------------------

duplicate_customer_ids = customers["customer_id"].duplicated().sum()

orders_without_customer = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

orders_customers_relation_ok = (
    duplicate_customer_ids == 0
    and orders_without_customer == 0
)

if orders_customers_relation_ok:
    orders_customers_result = "PASS (many_to_one)"
else:
    orders_customers_result = "FAIL"


# ---------------------------------------------------------
# 6. 실제 merge도 validate 옵션으로 관계 확인
# ---------------------------------------------------------

try:
    model_base = orders.merge(
        order_total,
        on="order_id",
        how="inner",
        validate="one_to_one"
    )

    model_base = model_base.merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_customer")
    )

    merge_validation = "PASS"

except pd.errors.MergeError as e:
    model_base = None
    merge_validation = f"FAIL: {e}"


# ---------------------------------------------------------
# 7. 검증 결과 출력
# ---------------------------------------------------------

print("===== STEP 3. Target 생성과 관계 검증 =====\n")

print("[Target]")
print("line_total = quantity × unit_price")
print("order_total = order_id별 line_total 합계")
print(f"생성된 Target 수: {len(order_total):,}건")

print("\n[line_total 검증]")
print(f"line_total 불일치 건수: {line_total_mismatch_count:,}건")

print("\n[orders ↔ target]")
print(f"orders의 order_id 중복: {duplicate_orders:,}건")
print(f"target의 order_id 중복: {duplicate_targets:,}건")
print(f"주문은 있지만 Target 없음: {orders_without_target:,}건")
print(f"Target은 있지만 부모 주문 없음: {targets_without_order:,}건")
print(f"관계 결과: {orders_target_result}")

print("\n[orders → customers]")
print(f"customers의 customer_id 중복: {duplicate_customer_ids:,}건")
print(f"customers에 없는 customer_id를 가진 주문: {orders_without_customer:,}건")
print(f"관계 결과: {orders_customers_result}")

print("\n[병합 validate]")
print(f"결과: {merge_validation}")


# ---------------------------------------------------------
# 8. Target 미리보기
# ---------------------------------------------------------

display(order_total.head())

===== STEP 3. Target 생성과 관계 검증 =====

[Target]
line_total = quantity × unit_price
order_total = order_id별 line_total 합계
생성된 Target 수: 301건

[line_total 검증]
line_total 불일치 건수: 0건

[orders ↔ target]
orders의 order_id 중복: 0건
target의 order_id 중복: 0건
주문은 있지만 Target 없음: 1건
Target은 있지만 부모 주문 없음: 2건
관계 결과: FAIL

[orders → customers]
customers의 customer_id 중복: 0건
customers에 없는 customer_id를 가진 주문: 2건
관계 결과: FAIL

[병합 validate]
결과: PASS


,order_id,order_total
0,1,344000.0
1,2,1389000.0
2,3,418000.0
3,4,116000.0
4,5,965000.0


In [5]:
# =========================================================
# STEP 3-1. 관계 오류 데이터 제거
# =========================================================

# 1. customers에 존재하는 고객의 주문만 유지
orders_clean = orders[
    orders["customer_id"].isin(customers["customer_id"])
].copy()

# 2. 실제 orders에 존재하는 주문 상세만 유지
order_items_clean = order_items[
    order_items["order_id"].isin(orders_clean["order_id"])
].copy()

# 3. line_total 다시 계산
order_items_clean["line_total_calc"] = (
    order_items_clean["quantity"]
    * order_items_clean["unit_price"]
)

# 4. 주문별 Target 다시 생성
order_total_clean = (
    order_items_clean
    .groupby("order_id", as_index=False)["line_total_calc"]
    .sum()
    .rename(columns={"line_total_calc": "order_total"})
)

# 5. Target이 존재하는 주문만 유지
orders_clean = orders_clean[
    orders_clean["order_id"].isin(order_total_clean["order_id"])
].copy()

# 6. 최종 모델링 데이터 생성
model_base = (
    orders_clean
    .merge(
        order_total_clean,
        on="order_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_customer")
    )
)

# 7. 최종 검증
print("===== 전처리 후 관계 검증 =====")

print(
    "주문은 있지만 Target 없음:",
    (~orders_clean["order_id"]
     .isin(order_total_clean["order_id"])).sum()
)

print(
    "Target은 있지만 부모 주문 없음:",
    (~order_total_clean["order_id"]
     .isin(orders_clean["order_id"])).sum()
)

print(
    "customers에 없는 customer_id:",
    (~orders_clean["customer_id"]
     .isin(customers["customer_id"])).sum()
)

print(f"\n최종 모델링 주문 수: {len(model_base):,}건")

display(model_base.head())

===== 전처리 후 관계 검증 =====
주문은 있지만 Target 없음: 0
Target은 있지만 부모 주문 없음: 0
customers에 없는 customer_id: 0

최종 모델링 주문 수: 297건


,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,name,gender,age,city,signup_date
0,1,144,2026-07-12,naver_pay,completed,2026-07,Sunday,344000.0,조우호,F,90.0,인천,2024-03-13
1,2,149,2025-10-02,kakao_pay,completed,2025-10,Thursday,1389000.0,김유람,M,32.0,서울,2025-03-23
2,3,34,2026-03-13,naver_pay,completed,2026-03,Friday,418000.0,윤승수,M,54.0,광주,2026-06-09
3,4,92,2025-10-12,kakao_pay,completed,2025-10,Sunday,116000.0,강영원,F,25.0,광주,2025-01-04
4,5,6,2026-04-17,bank_transfer,cancelled,2026-04,Friday,965000.0,임정우,M,27.0,대구,2023-07-21


### Target 생성과 관계 검증

- `line_total = quantity × unit_price`
- `order_total = order_id별 line_total 합계`
- 기존 `line_total`과 재계산 값의 불일치: **0건**
- Target이 없는 주문: **1건**
- 부모 주문이 없는 Target: **2건**
- 존재하지 않는 고객을 참조하는 주문: **2건**

관계가 맞지 않는 소수의 데이터는 임의로 값을 수정하지 않고 모델링 대상에서 제외하였다.

전처리 후 `orders ↔ target`은 **one-to-one**, `orders → customers`는 **many-to-one** 관계가 성립하는 데이터만 사용한다.

또한 `quantity`, `unit_price`, `line_total`은 Target 생성에만 사용하고 모델의 Feature에는 포함하지 않는다.

STEP 4는 과거 데이터로 공부한 AI가 미래 데이터를 얼마나 잘 예측하는지 시험하기 위한 단계입니다.
시험공부로 비유하면 Train은 공부할 문제, Final Test는 실제 시험 문제입니다.
과거 주문은 Train으로, 가장 최근 주문은 Final Test로 나눕니다.
랜덤으로 섞으면 미래의 정보를 미리 공부하게 되어 실제 상황과 맞지 않을 수 있습니다.
그래서 과거로 학습 → 미래를 예측하는 실제 상황과 비슷하게 데이터를 나누는 것입니다.


실제 서비스에서는 과거 데이터를 이용해 미래의 주문 금액을 예측하기 때문에, 실제 예측 상황과 비슷하게 만들기 위해 랜덤 분할이 아닌 시간 순서 분할을 사용한다.

In [15]:
# =========================================================
# STEP 4. Train / Final Test 시간 순서 분리
# =========================================================

import pandas as pd
from pathlib import Path


# 1. 날짜형으로 변환
model_base["order_date"] = pd.to_datetime(
    model_base["order_date"],
    errors="coerce"
)

# 날짜가 없는 행은 시간 분할이 불가능하므로 제외
model_base = model_base.dropna(subset=["order_date"]).copy()


# 2. 시간 순서대로 정렬
model_base = (
    model_base
    .sort_values("order_date")
    .reset_index(drop=True)
)


# 3. 마지막 약 20%가 시작되는 날짜 찾기
split_index = int(len(model_base) * 0.8)

split_date = model_base.loc[
    split_index,
    "order_date"
]


# 4. 같은 날짜가 양쪽에 들어가지 않도록 날짜 기준 분리
train_data = model_base[
    model_base["order_date"] < split_date
].copy()

final_test = model_base[
    model_base["order_date"] >= split_date
].copy()


# 5. 날짜 overlap 확인
train_dates = set(train_data["order_date"].dt.date)
test_dates = set(final_test["order_date"].dt.date)

date_overlap = train_dates.intersection(test_dates)


# 6. 결과 요약
split_summary = pd.DataFrame({
    "구분": ["Train", "Final Test"],
    "시작일": [
        train_data["order_date"].min(),
        final_test["order_date"].min()
    ],
    "종료일": [
        train_data["order_date"].max(),
        final_test["order_date"].max()
    ],
    "행 수": [
        len(train_data),
        len(final_test)
    ]
})

display(split_summary)

print("===== STEP 4. 시간 순서 분리 검증 =====")
print(f"분리 기준 날짜: {split_date.date()}")

print(
    "Train 최대 날짜:",
    train_data["order_date"].max().date()
)

print(
    "Final Test 최소 날짜:",
    final_test["order_date"].min().date()
)

print(
    "같은 날짜 overlap:",
    len(date_overlap)
)

print(
    "시간 순서 조건:",
    train_data["order_date"].max()
    < final_test["order_date"].min()
)


# 7. Evidence CSV 저장
report_path = Path("../reports")
report_path.mkdir(parents=True, exist_ok=True)

split_summary.to_csv(
    report_path / "ch09_regression_split_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료:")
print("../reports/ch09_regression_split_summary.csv")

,구분,시작일,종료일,행 수
0,Train,2025-07-01,2026-05-27,235
1,Final Test,2026-05-30,2028-12-31,59


===== STEP 4. 시간 순서 분리 검증 =====
분리 기준 날짜: 2026-05-30
Train 최대 날짜: 2026-05-27
Final Test 최소 날짜: 2026-05-30
같은 날짜 overlap: 0
시간 순서 조건: True

저장 완료:
../reports/ch09_regression_split_summary.csv


### STEP 5. 전처리는 Pipeline 내부에서 수행합니다

## 전처리 누수 방지

- **숫자형 처리:** 결측값을 중앙값(median)으로 채운 뒤 StandardScaler로 크기를 맞춘다.
- **범주형 처리:** 결측값을 가장 자주 나온 값으로 채운 뒤 OneHotEncoder로 숫자 형태로 변환한다.
- **전처리 fit 범위:** Train 데이터에서만 전처리 방법을 학습하고, Validation과 Final Test에는 학습된 방법을 적용만 한다.
- **전체 데이터에 먼저 fit하면 안 되는 이유:** Test 데이터의 정보를 미리 사용하게 되어 실제보다 모델 성능이 좋게 측정되는 데이터 누수가 발생할 수 있기 때문이다.

In [10]:
# =========================================================
# STEP 5. 전처리 Pipeline 구성
# =========================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# 1. 사용할 Feature 구분
# 숫자형 Feature
numeric_features = [
    "age"
]

# 범주형 Feature
categorical_features = [
    "gender",
    "city",
    "payment_method",
    "order_dayofweek",
    "order_month"
]


# 2. 숫자형 전처리
# 결측치 → 중앙값 → 표준화
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


# 3. 범주형 전처리
# 결측치 → 최빈값 → One-Hot Encoding
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# 4. 숫자형 + 범주형 전처리 결합
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# 5. 확인
print("===== STEP 5. 전처리 Pipeline =====")
print("숫자형 Feature:", numeric_features)
print("범주형 Feature:", categorical_features)

print("\n숫자형 처리")
print("결측치 → median → StandardScaler")

print("\n범주형 처리")
print("결측치 → most_frequent → OneHotEncoder")

print("\n✅ 전처리 Pipeline 준비 완료")
print("※ 아직 전체 데이터에 fit하지 않았습니다.")

===== STEP 5. 전처리 Pipeline =====
숫자형 Feature: ['age']
범주형 Feature: ['gender', 'city', 'payment_method', 'order_dayofweek', 'order_month']

숫자형 처리
결측치 → median → StandardScaler

범주형 처리
결측치 → most_frequent → OneHotEncoder

✅ 전처리 Pipeline 준비 완료
※ 아직 전체 데이터에 fit하지 않았습니다.


### STEP 6. Baseline과 후보 모델을 준비합니다
## Baseline과 후보 모델

- **Baseline Mean:** 모든 주문 금액을 Train 데이터의 평균값으로 예측하는 기준 모델
- **Linear Regression:** Feature와 주문 금액의 선형적인 관계를 학습하는 모델
- **Random Forest:** 여러 개의 결정 트리를 이용해 복잡한 관계를 학습하는 모델
- **Baseline을 사용하는 이유:** 복잡한 모델이 단순히 평균을 예측하는 것보다 실제로 더 좋은 성능을 내는지 비교하기 위해 사용한다.
- **판단:** Random Forest가 더 복잡하다는 이유만으로 우수하다고 할 수 없으며, 실제 검증 결과를 비교해야 한다.

In [12]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# STEP 6. Baseline 및 후보 모델 준비
# ==============================================================================
models = {
    "Baseline Mean": DummyRegressor(strategy="mean"), # 기준점 (단순 평균)
    "Linear Regression": Ridge(alpha=1.0, random_state=42), # 비베이스라인 후보 1
    "Random Forest": RandomForestRegressor(random_state=42) # 비베이스라인 후보 2
}

STEP 7. Train 기간에서 TimeSeriesSplit으로 후보를 비교합니다

In [16]:
# =========================================================
# STEP 7. TimeSeriesSplit으로 후보 모델 비교
# =========================================================

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


# ---------------------------------------------------------
# 1. 사용할 Feature / Target
# ---------------------------------------------------------

feature_columns = [
    "age",
    "gender",
    "city",
    "payment_method",
    "order_dayofweek",
    "order_month"
]

target_column = "order_total"

# 시간 순서 보장
train_data = train_data.sort_values("order_date").reset_index(drop=True)

X_train = train_data[feature_columns]
y_train = train_data[target_column]


# ---------------------------------------------------------
# 2. 후보 모델 준비
# ---------------------------------------------------------

models = {
    "Baseline Mean": DummyRegressor(strategy="mean"),

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}


# ---------------------------------------------------------
# 3. TimeSeriesSplit 준비
# ---------------------------------------------------------

tscv = TimeSeriesSplit(n_splits=5)


# ---------------------------------------------------------
# 4. 모델별 교차검증
# ---------------------------------------------------------

results = []

for model_name, model in models.items():

    # STEP 5에서 만든 전처리 + 모델 연결
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=tscv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    # sklearn은 MAE를 음수로 반환하므로 다시 양수로 변환
    mae_scores = -scores["test_MAE"]
    r2_scores = scores["test_R2"]

    results.append({
        "모델": model_name,
        "cv_MAE_mean": mae_scores.mean(),
        "cv_MAE_std": mae_scores.std(),
        "cv_R2_mean": r2_scores.mean()
    })


# ---------------------------------------------------------
# 5. 결과표 생성
# ---------------------------------------------------------

cv_summary = pd.DataFrame(results)

display(
    cv_summary.style.format({
        "cv_MAE_mean": "{:,.2f}",
        "cv_MAE_std": "{:,.2f}",
        "cv_R2_mean": "{:.4f}"
    })
)


# ---------------------------------------------------------
# 6. Linear Regression vs Random Forest 중 선택
# ---------------------------------------------------------

candidate_results = cv_summary[
    cv_summary["모델"].isin([
        "Linear Regression",
        "Random Forest"
    ])
]

best_model_name = (
    candidate_results
    .sort_values("cv_MAE_mean")
    .iloc[0]["모델"]
)

print("===== 모델 선택 결과 =====")
print(f"선택 모델: {best_model_name}")
print("선택 기준: Train CV MAE 평균이 더 낮은 모델")


# ---------------------------------------------------------
# 7. Evidence CSV 저장
# ---------------------------------------------------------

report_path = Path("../reports")
report_path.mkdir(parents=True, exist_ok=True)

cv_summary.to_csv(
    report_path / "ch09_regression_cv_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료:")
print("../reports/ch09_regression_cv_summary.csv")

,모델,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,"450,511.70","43,914.32",-0.1854
1,Linear Regression,"791,051.91","252,974.50",-12.8334
2,Random Forest,"642,687.93","118,909.91",-6.0452


===== 모델 선택 결과 =====
선택 모델: Random Forest
선택 기준: Train CV MAE 평균이 더 낮은 모델

저장 완료:
../reports/ch09_regression_cv_summary.csv


## TimeSeriesSplit 후보 모델 비교 결과

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
|---|---:|---:|---:|---|
| Baseline Mean | 450,511.70 | 43,914.32 | -0.1854 | 기준 모델 |
| Linear Regression | 791,051.91 | 252,974.50 | -12.8334 | 오차가 크고 fold별 성능 변화도 큼 |
| Random Forest | 642,687.93 | 118,909.91 | -6.0452 | Linear Regression보다 MAE가 낮아 후보 모델로 선택 |

### 결과 해석

- 비베이스라인 모델 중 **Random Forest의 CV MAE 평균이 더 낮아 최종 후보 모델로 선택**하였다.
- Linear Regression은 MAE 평균뿐만 아니라 표준편차도 가장 커서 fold에 따라 성능이 크게 흔들렸다.
- Random Forest도 표준편차가 118,909.91로 나타나 기간에 따라 성능 차이가 존재한다.
- 모든 후보 모델의 CV R²가 음수이므로 현재 Feature만으로 주문 금액을 충분히 설명하지 못하고 있다.
- 특히 **Baseline Mean의 MAE가 두 후보 모델보다 낮다.** 따라서 Random Forest가 Linear Regression보다는 좋지만, 단순 평균 예측보다 좋은 성능을 보였다고 할 수는 없다.

### 모델 선택

실습 기준에 따라 Linear Regression과 Random Forest 중 CV MAE 평균이 더 낮은 **Random Forest를 선택**한다.

다만 Baseline Mean보다 MAE가 높으므로, 현재 모델의 실제 예측 성능에는 한계가 있다는 점을 함께 기록한다.

## Train CV로 선택한 모델

- **Selected Model:** Random Forest
- **선택 기준:** 비베이스라인 후보인 Linear Regression과 Random Forest 중 Train CV의 `cv_MAE_mean`이 더 낮은 모델을 선택
- **근거가 된 cv_MAE_mean:** 642,687.93
- **Final Test를 보기 전에 선택했는가:** 예

Random Forest의 CV MAE 평균은 642,687.93으로 Linear Regression의 791,051.91보다 낮았다.  
따라서 Train 데이터의 교차검증 결과만을 기준으로 Random Forest를 Selected Model로 고정한다.

※ Baseline Mean의 CV MAE 평균(450,511.70)이 더 낮았지만, 이번 실습의 모델 선택 기준은 비베이스라인 후보인 Linear Regression과 Random Forest를 비교하는 것이다.

**모델 선택은 여기서 종료하며, 이후 Final Test 결과를 보고 모델을 변경하지 않는다.**

STEP 9. Final Test에서는 Baseline과 Frozen Model만 평가합니다

 

이제 처음으로 Final Test를 사용합니다.

In [17]:
# =========================================================
# STEP 9. Final Test 평가
# Baseline Mean vs Frozen Selected Model (Random Forest)
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ---------------------------------------------------------
# 1. Feature / Target 준비
# ---------------------------------------------------------

feature_columns = [
    "age",
    "gender",
    "city",
    "payment_method",
    "order_dayofweek",
    "order_month"
]

target_column = "order_total"

X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_test = final_test[feature_columns]
y_test = final_test[target_column]


# ---------------------------------------------------------
# 2. 평가할 모델
# STEP 8에서 선택한 Random Forest를 그대로 고정
# ---------------------------------------------------------

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyRegressor(strategy="mean"))
    ]
)

selected_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


# ---------------------------------------------------------
# 3. Train 데이터로만 학습
# ---------------------------------------------------------

baseline_pipeline.fit(X_train, y_train)
selected_pipeline.fit(X_train, y_train)


# ---------------------------------------------------------
# 4. Train / Final Test 예측
# ---------------------------------------------------------

baseline_train_pred = baseline_pipeline.predict(X_train)
baseline_test_pred = baseline_pipeline.predict(X_test)

selected_train_pred = selected_pipeline.predict(X_train)
selected_test_pred = selected_pipeline.predict(X_test)


# ---------------------------------------------------------
# 5. 평가 함수
# ---------------------------------------------------------

def evaluate_model(
    model_name,
    selection_role,
    y_train,
    train_pred,
    y_test,
    test_pred
):
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    return {
        "모델": model_name,
        "selection_role": selection_role,
        "train_MAE": train_mae,
        "test_MAE": test_mae,
        "test_RMSE": test_rmse,
        "test_R2": test_r2
    }


# ---------------------------------------------------------
# 6. Baseline / Frozen Model 평가
# ---------------------------------------------------------

results = [
    evaluate_model(
        "Baseline Mean",
        "baseline",
        y_train,
        baseline_train_pred,
        y_test,
        baseline_test_pred
    ),

    evaluate_model(
        "Frozen Selected Model (Random Forest)",
        "selected_by_train_cv",
        y_train,
        selected_train_pred,
        y_test,
        selected_test_pred
    )
]

comparison = pd.DataFrame(results)


# ---------------------------------------------------------
# 7. Baseline 대비 MAE 개선율 계산
#
# 양수  → Baseline보다 개선
# 음수  → Baseline보다 성능 저하
# ---------------------------------------------------------

baseline_test_mae = comparison.loc[
    comparison["모델"] == "Baseline Mean",
    "test_MAE"
].iloc[0]

comparison["MAE_improvement_vs_baseline_pct"] = (
    (baseline_test_mae - comparison["test_MAE"])
    / baseline_test_mae
    * 100
)

# Baseline 자신은 비교 기준이므로 0%
comparison.loc[
    comparison["모델"] == "Baseline Mean",
    "MAE_improvement_vs_baseline_pct"
] = 0.0


# ---------------------------------------------------------
# 8. 결과 출력
# ---------------------------------------------------------

display(
    comparison.style.format({
        "train_MAE": "{:,.2f}",
        "test_MAE": "{:,.2f}",
        "test_RMSE": "{:,.2f}",
        "test_R2": "{:.4f}",
        "MAE_improvement_vs_baseline_pct": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 9. Evidence CSV 저장
# ---------------------------------------------------------

report_path = Path("../reports")
report_path.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    report_path / "ch09_regression_model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("===== STEP 9. Final Test 평가 완료 =====")
print(f"Train 데이터: {len(X_train):,}건")
print(f"Final Test 데이터: {len(X_test):,}건")
print("Frozen Selected Model: Random Forest")

print("\n저장 완료:")
print("../reports/ch09_regression_model_comparison.csv")

,모델,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,"488,948.12","477,215.65","563,314.03",-0.0004,0.00%
1,Frozen Selected Model (Random Forest),selected_by_train_cv,"207,953.53","495,116.82","605,680.94",-0.1565,-3.75%


===== STEP 9. Final Test 평가 완료 =====
Train 데이터: 235건
Final Test 데이터: 59건
Frozen Selected Model: Random Forest

저장 완료:
../reports/ch09_regression_model_comparison.csv


## Final Test 평가 결과

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
|---|---|---:|---:|---:|---:|---:|
| Baseline Mean | baseline | 488,948.12 | 477,215.65 | 563,314.03 | -0.0004 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 207,953.53 | 495,116.82 | 605,680.94 | -0.1565 | -3.75% |

### 결과 해석

- Train CV에서 선택하여 고정한 **Random Forest**를 Final Test에서 평가하였다.
- Random Forest의 Test MAE는 **495,116.82**로 Baseline Mean의 **477,215.65**보다 높았다.
- Baseline 대비 MAE 개선율은 **-3.75%**로, Random Forest가 Baseline보다 약 3.75% 더 좋지 않은 결과를 보였다.
- Random Forest는 Train MAE가 207,953.53으로 낮지만 Test MAE는 크게 증가하여, 학습 데이터에 비해 새로운 데이터에서 성능이 떨어지는 모습을 보였다.
- Test R²도 **-0.1565**로 나타나 현재 Feature만으로는 미래 주문 금액을 충분히 설명하지 못했다.

### 최종 판단

Random Forest는 Train CV를 기준으로 미리 선택한 모델이므로 Final Test 결과가 좋지 않더라도 다른 모델로 교체하지 않는다.

이번 결과에서는 복잡한 Random Forest가 단순 평균을 예측하는 Baseline보다 더 좋은 성능을 보이지 못했다. 따라서 현재 Feature 또는 모델링 방법에 개선이 필요하다고 판단할 수 있다.

## Final Test 해석

### 결과 관찰

- **MAE:** 495,116.82원
- **RMSE:** 605,680.94원
- **R²:** -0.1565
- **Baseline 대비 개선율:** -3.75%

### 나의 해석과 판단

Random Forest는 주문 금액을 예측할 때 실제 금액과 **평균적으로 약 49만 5천 원 정도의 오차**가 발생했다.

RMSE는 약 60만 6천 원으로 MAE보다 높게 나타났다. RMSE는 큰 오차에 더 큰 영향을 받기 때문에, 일부 주문에서 예측값과 실제값의 차이가 크게 발생했을 가능성이 있다.

R²는 -0.1565로 0보다 낮았다. 이는 코드 오류를 의미하는 것이 아니라, Final Test에서는 Random Forest가 단순히 평균값을 예측하는 것보다 주문 금액의 변화를 잘 설명하지 못했다는 의미이다.

### 업무·분석적 의미

Baseline Mean의 Test MAE는 477,215.65원으로 Random Forest보다 낮았다. Random Forest의 Baseline 대비 MAE 개선율도 -3.75%이므로, 현재 모델을 실제 주문 금액 예측에 활용했을 때 단순 평균 예측보다 추가적인 이점을 얻었다고 보기 어렵다.

### 현재 한계

현재 사용한 Feature만으로는 주문 금액을 충분히 설명하기 어려운 것으로 보인다. 또한 Random Forest의 Train MAE는 207,953.53원이지만 Test MAE는 495,116.82원으로 차이가 크기 때문에, 학습 데이터에서는 잘 맞았지만 새로운 미래 데이터에서는 성능이 떨어지는 현상도 확인할 필요가 있다.

따라서 향후에는 Feature 구성, 데이터 품질, 시간에 따른 데이터 변화 등을 추가로 확인할 필요가 있다.

STEP 11. 대표 오차 사례를 진단합니다

In [18]:
# =========================================================
# STEP 11. 대표 오차 사례 진단
# Frozen Selected Model: Random Forest
# =========================================================

import pandas as pd
from pathlib import Path


# 1. Final Test 예측 결과 생성
predictions = pd.DataFrame({
    "order_id": final_test["order_id"].values,
    "actual_order_total": y_test.values,
    "predicted_order_total": selected_test_pred
})


# 2. residual / absolute error 계산
# residual = 실제값 - 예측값
predictions["residual"] = (
    predictions["actual_order_total"]
    - predictions["predicted_order_total"]
)

predictions["abs_error"] = predictions["residual"].abs()


# 3. 오차가 큰 순서대로 정렬
predictions = (
    predictions
    .sort_values("abs_error", ascending=False)
    .reset_index(drop=True)
)


# 4. 대표 오차 사례 3개
top_errors = predictions.head(3)

print("===== STEP 11. 대표 오차 사례 =====\n")

display(
    top_errors.style.format({
        "actual_order_total": "{:,.2f}",
        "predicted_order_total": "{:,.2f}",
        "residual": "{:,.2f}",
        "abs_error": "{:,.2f}"
    })
)


# 5. residual 방향 설명
for i, row in top_errors.iterrows():

    if row["residual"] > 0:
        direction = "실제보다 낮게 예측"
    elif row["residual"] < 0:
        direction = "실제보다 높게 예측"
    else:
        direction = "정확히 예측"

    print(f"사례 {i + 1}")
    print(f"실제값: {row['actual_order_total']:,.2f}원")
    print(f"예측값: {row['predicted_order_total']:,.2f}원")
    print(f"residual: {row['residual']:,.2f}원")
    print(f"absolute error: {row['abs_error']:,.2f}원")
    print(f"관찰: {direction}")
    print()


# 6. 내부 진단용 CSV 저장
report_path = Path("../reports")
report_path.mkdir(parents=True, exist_ok=True)

predictions.to_csv(
    report_path / "ch09_regression_predictions_internal.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:")
print("../reports/ch09_regression_predictions_internal.csv")

===== STEP 11. 대표 오차 사례 =====



,order_id,actual_order_total,predicted_order_total,residual,abs_error
0,163,"18,000.00","1,474,180.00","-1,456,180.00","1,456,180.00"
1,129,"1,791,000.00","537,213.33","1,253,786.67","1,253,786.67"
2,231,"230,000.00","1,474,180.00","-1,244,180.00","1,244,180.00"


사례 1
실제값: 18,000.00원
예측값: 1,474,180.00원
residual: -1,456,180.00원
absolute error: 1,456,180.00원
관찰: 실제보다 높게 예측

사례 2
실제값: 1,791,000.00원
예측값: 537,213.33원
residual: 1,253,786.67원
absolute error: 1,253,786.67원
관찰: 실제보다 낮게 예측

사례 3
실제값: 230,000.00원
예측값: 1,474,180.00원
residual: -1,244,180.00원
absolute error: 1,244,180.00원
관찰: 실제보다 높게 예측

저장 완료:
../reports/ch09_regression_predictions_internal.csv


## 대표 오차 사례 진단

### 사례 1

- **실제값:** 18,000원
- **예측값:** 1,474,180원
- **residual:** -1,456,180원
- **absolute error:** 1,456,180원
- **관찰:** 실제 주문 금액보다 약 145만 원 높게 예측했다.
- **가능한 이유 후보:** 현재 Feature만으로 매우 낮은 금액의 주문을 다른 주문과 충분히 구분하지 못했을 가능성이 있다.
- **추가 확인 사항:** 해당 주문의 고객 특성, 주문 시점 등이 다른 주문과 어떤 차이가 있는지 확인할 필요가 있다.

### 사례 2

- **실제값:** 1,791,000원
- **예측값:** 537,213.33원
- **residual:** 1,253,786.67원
- **absolute error:** 1,253,786.67원
- **관찰:** 실제 주문 금액보다 약 125만 원 낮게 예측했다.
- **가능한 이유 후보:** 현재 Feature가 고액 주문의 특징을 충분히 설명하지 못했을 가능성이 있다.
- **추가 확인 사항:** 해당 주문이 다른 주문보다 특별히 높은 금액을 가지게 된 특징이 있는지 추가 확인이 필요하다.

### 사례 3

- **실제값:** 230,000원
- **예측값:** 1,474,180원
- **residual:** -1,244,180원
- **absolute error:** 1,244,180원
- **관찰:** 실제 주문 금액보다 약 124만 원 높게 예측했다.
- **가능한 이유 후보:** 현재 Feature만으로 주문 금액이 낮은 주문을 충분히 구분하지 못했을 가능성이 있다.
- **추가 확인 사항:** 사례 1과 예측값이 동일하므로 두 주문의 입력 Feature가 서로 비슷하거나 동일한지 확인할 필요가 있다.

### 종합 판단

대표 오차 사례에서는 저액 주문을 크게 높게 예측하거나 고액 주문을 낮게 예측하는 현상이 나타났다.

특히 사례 1과 사례 3은 실제 금액이 크게 다른데도 **1,474,180원이라는 동일한 예측값**이 나왔다. 따라서 현재 사용한 Feature가 주문 금액 차이를 충분히 설명하지 못하고 있을 가능성이 있다.

다만 현재 결과만으로 정확한 원인을 단정할 수 없으므로, 해당 주문들의 Feature와 데이터 특성을 추가로 확인해야 한다.

STEP 12. 자동 Validation Evidence를 확인합니다

In [19]:
# =========================================================
# STEP 12. 자동 Validation Evidence 생성 및 확인
# =========================================================

import pandas as pd
from pathlib import Path


# ---------------------------------------------------------
# 1. 금지 Feature가 모델 입력에 포함되었는지 확인
# ---------------------------------------------------------

forbidden_features = {
    "order_total",
    "line_total",
    "quantity",
    "unit_price",
    "price",
    "order_id",
    "customer_id",
    "product_id",
    "order_item_id",
    "order_status",
    "name",
    "product_name",
    "order_match",
    "product_match",
    "customer_match",
    "merge_valid"
}

overlap = sorted(set(feature_columns) & forbidden_features)

check_forbidden = len(overlap) == 0


# ---------------------------------------------------------
# 2. Train이 Final Test보다 완전히 과거인지 확인
# ---------------------------------------------------------

train_max_date = train_data["order_date"].max()
test_min_date = final_test["order_date"].min()

check_time = train_max_date < test_min_date


# ---------------------------------------------------------
# 3. Selected Model이 Train CV에 존재하는지 확인
# ---------------------------------------------------------

selected_model_name = "Random Forest"

check_selected_cv = (
    selected_model_name
    in cv_summary["모델"].values
)


# ---------------------------------------------------------
# 4. Final Test 결과에 Baseline이 있는지 확인
# ---------------------------------------------------------

check_baseline = (
    comparison["selection_role"]
    .eq("baseline")
    .any()
)


# ---------------------------------------------------------
# 5. Final Test 결과에 Frozen Selected Model이 있는지 확인
# ---------------------------------------------------------

check_frozen = (
    comparison["selection_role"]
    .eq("selected_by_train_cv")
    .any()
)


# ---------------------------------------------------------
# 6. R² 계산에 충분한 Test 행이 있는지 확인
# 최소 2개 이상의 행 필요
# ---------------------------------------------------------

test_row_count = len(final_test)

check_r2 = test_row_count >= 2


# ---------------------------------------------------------
# 7. Validation 결과표 생성
# ---------------------------------------------------------

validation = pd.DataFrame([
    {
        "check": "forbidden_feature_overlap",
        "value": (
            "없음"
            if check_forbidden
            else ", ".join(overlap)
        ),
        "status": "PASS" if check_forbidden else "FAIL"
    },
    {
        "check": "strict_train_before_test",
        "value": (
            f"{train_max_date.date()} < "
            f"{test_min_date.date()}"
        ),
        "status": "PASS" if check_time else "FAIL"
    },
    {
        "check": "selected_model_exists_in_train_cv",
        "value": selected_model_name,
        "status": "PASS" if check_selected_cv else "FAIL"
    },
    {
        "check": "final_test_contains_baseline",
        "value": str(check_baseline),
        "status": "PASS" if check_baseline else "FAIL"
    },
    {
        "check": "final_test_contains_frozen_selected_model",
        "value": str(check_frozen),
        "status": "PASS" if check_frozen else "FAIL"
    },
    {
        "check": "test_rows_for_r2",
        "value": str(test_row_count),
        "status": "PASS" if check_r2 else "FAIL"
    }
])


# ---------------------------------------------------------
# 8. 결과 출력
# ---------------------------------------------------------

display(validation)

print("===== STEP 12. Validation Evidence =====")

if (validation["status"] == "PASS").all():
    print("✅ 모든 Validation 검사 PASS")
    print("Chapter 09 모델링 완료 조건을 충족했습니다.")
else:
    print("❌ FAIL 항목이 있습니다.")
    print("아래 항목을 수정한 뒤 다시 검증해야 합니다.")

    display(
        validation[
            validation["status"] == "FAIL"
        ]
    )


# ---------------------------------------------------------
# 9. Evidence CSV 저장
# ---------------------------------------------------------

report_path = Path("../reports")
report_path.mkdir(parents=True, exist_ok=True)

validation.to_csv(
    report_path / "ch09_regression_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료:")
print("../reports/ch09_regression_validation.csv")

,check,value,status
0,forbidden_feature_overlap,없음,PASS
1,strict_train_before_test,2026-05-27 < 2026-05-30,PASS
2,selected_model_exists_in_train_cv,Random Forest,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,59,PASS


===== STEP 12. Validation Evidence =====
✅ 모든 Validation 검사 PASS
Chapter 09 모델링 완료 조건을 충족했습니다.

저장 완료:
../reports/ch09_regression_validation.csv


## 자동 Validation Evidence 확인

| check | value | status |
|---|---|---|
| forbidden_feature_overlap | 없음 | PASS |
| strict_train_before_test | 2026-05-27 < 2026-05-30 | PASS |
| selected_model_exists_in_train_cv | Random Forest | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 59 | PASS |

### 검증 결과

- 금지된 Feature가 모델 입력에 포함되지 않았다.
- Train의 마지막 날짜는 2026-05-27, Final Test의 시작 날짜는 2026-05-30으로 시간 순서가 올바르게 분리되었다.
- Train CV에서 선택한 Random Forest가 정상적으로 기록되어 있다.
- Final Test에는 Baseline과 Frozen Selected Model만 포함되어 있다.
- Final Test는 59건으로 R²를 계산하기에 충분한 데이터가 있다.
- **모든 Validation 항목이 PASS이므로 Chapter 09 모델링 완료 조건을 충족했다.**

### FAIL 원인

없음

### 수정 내용

추가 수정 없음

### 재검증 결과

모든 항목 PASS

STEP 13. Evidence와 공개 범위를 구분합니다

## Evidence와 공개 범위

### 공개 가능한 Evidence

다음 파일은 모델링 과정과 검증 결과를 보여 주는 자료로 공개 가능하다.

- `reports/ch09_regression_split_summary.csv`
- `reports/ch09_regression_feature_audit.csv`
- `reports/ch09_regression_cv_summary.csv`
- `reports/ch09_regression_model_comparison.csv`
- `reports/ch09_regression_validation.csv`
- `reports/ch09_regression_checklist.csv`
- `reports/ch09_regression_report.md`
- `reports/figures/ch09_actual_vs_predicted.png`
- `reports/figures/ch09_residual_histogram.png`

### Internal로만 유지할 파일

- `reports/ch09_regression_model_data_internal.csv`
- `reports/ch09_regression_predictions_internal.csv`

### Internal로 구분한 이유

Internal 파일에는 `order_id`와 같은 원본 식별자가 포함될 수 있다.

이러한 정보는 모델의 오차 사례를 추적하고 내부적으로 분석할 때는 필요하지만,
외부에 공개하는 분석 결과에는 반드시 필요한 정보가 아니다.

따라서 공개용 보고서와 그래프에서는 분석에 불필요한 식별자를 제거하고,
상세 데이터와 식별자가 포함된 파일은 Internal 자료로만 유지한다.

### 최종 판단

모델링 과정이 올바른지 검증하는 것과 결과를 어디까지 공개할 수 있는지
검토하는 것은 서로 다른 문제이다.

따라서 모델 검증을 통과했더라도 공개 전에 식별자나 불필요한 상세 정보가
포함되어 있지 않은지 별도로 확인해야 한다.